# Scene completion
The pipeline to complete the partial environment scan.

In [ ]:
import numpy as np
import open3d as o3d
import trimesh
import os
import sys
sys.path.insert(0, '../')
import drm
import drm.detect
import drm.align
import drm.generate
import drm.pipeline
from PIL import Image
import torch
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

## V-Scan Scene Completion
Perform the scene completion on the V-Scan dataset by:
- Loading the pointcloud
- loading the object Boundingboxes
- loading the scene occlusions
- Cutting out the bounding boxes
- Detecting the planes
- Finding plane intersections
- Isolating the points per plane
- filling in the parts between the intersections and in the occlusion regions
- Painting in the missing textures

### Data Loading

In [ ]:
scene_dir = Path(r"/home/jvermandere/datasets/V-Scan/data/Industrial_1_Leica-P30_1775813780764")

SCAN_FILE = "main.txt"
PANO_FILE = "pano.png"
EMPTY_PANO_FILE = "pano_empty.png"
BB_FILE = "main_bb.json"
OCCLUSION_FILE = "occluded_grid.ply"
VOXEL_SIZE = 0.05

pcd, matrix = drm.txt_pcd_to_open3d(scene_dir / SCAN_FILE)
#pcd.translate(matrix[:3, 3])
pcd = pcd.voxel_down_sample(VOXEL_SIZE)
trans_matrix = drm.read_transform_matrix(scene_dir / SCAN_FILE, apply_unity_conversion=True)
translate_matrix = np.eye(4)
translate_matrix[:3, 3] = trans_matrix[:3, 3]

boxes = drm.trimesh_to_open3d(drm.detect.load_gt_json_boxes_as_mesh(scene_dir / BB_FILE))
expanded_boxes = drm.expand_mesh(boxes, offset=0.1)

occlusion_points = o3d.io.read_point_cloud(scene_dir / OCCLUSION_FILE)
occlusion_grid = drm.load_voxelgrid_from_ply(scene_dir / OCCLUSION_FILE, transform=translate_matrix)
scene = drm.visualise_open3d([pcd] + expanded_boxes + [occlusion_grid], voxel_scale_modifier=0.9)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

### Bounding box cutout

This is part of the last chapter, here the boundingboxes are just cut out of the pointcloud to get clean cutouts.

In [ ]:
isolated_pcds, remainder_pcd = drm.detect.split_pointcloud_by_boxes(pcd, expanded_boxes)

scene = drm.visualise_open3d(isolated_pcds, random_color=True)
scene.add_geometry(drm.o3d_pointcloud_to_trimesh(remainder_pcd))
scene.show()

### Plane Detection

Already in the last chapter, but used as a starting point for the scene completion chapter.
The planes are itteratively detected, and stopped once no more planes are found with a high enough pointcount

In [ ]:
detected_planes, plane_models, leftovers = drm.detect.detect_planes_iteratively(remainder_pcd,min_points=100, num_iterations=1000, distance_threshold=0.1)

drm.visualise_open3d(detected_planes, random_color=True).show()

### Plane boundary detection

To get clean edges and connecting planes we create oversized plane meshes from the detected planes. The we cut all the planes with eachother to get a bunch of seperate planes. Then we use the remaining pointcloud to only keep the planes that have a sufficient amount of points. Furthermore the points are also segmented by the planes, so each plane has their corresponding points

In [ ]:

plane_meshes = drm.generate.create_plane_meshes(plane_models, pcd.get_axis_aligned_bounding_box(), max_extend_modifier=1)
print(f"Detected {len(plane_meshes)} planes after clipping.")
scene = drm.visualise_open3d(plane_meshes + [pcd], random_color=True)
scene.show()

In [ ]:
filtered_meshes, filtered_pcds = drm.generate.filter_planes_by_points(plane_meshes, detected_planes, distance_threshold=0.05, min_points=500)

scene = drm.visualise_open3d(filtered_meshes +  filtered_pcds, random_color=True)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

### New Point generation

Next we need to fill in the missing regions of the pointcoud. but we want a mesh as endresult. so we sample points on the plane with a similar density as the surrounding points. Then we subdivide the plane into the same sensity. then we project the plane along its normal direction to the closest pointcloud point, except for the edges. to ensure a clean connection with the surrounding planes. We assume the room is a closed space so all the planes connect. openings such as doors and windows are handled in a later chapter (8) where they will be made as dynamic entities that can be moved and scaled along the walls 

In [ ]:
unoccupied = drm.generate.sample_unoccupied_plane_points(filtered_meshes, filtered_pcds, voxel_size=VOXEL_SIZE)

combined_pcds = [
    pcd + unoccupied_pcd
    for pcd, unoccupied_pcd in zip(filtered_pcds, unoccupied)
]

# Visualise occupied vs unoccupied side by side
scene = drm.visualise_open3d(filtered_pcds, random_color=True)
for pcd in unoccupied:
    tm = drm.o3d_pointcloud_to_trimesh(pcd)
    if tm is None:
        continue
    tm.colors = np.full((len(tm.vertices), 4), [255, 50, 50, 255], dtype=np.uint8)
    scene.add_geometry(tm)
scene.show()

In [ ]:
projected_meshes = drm.generate.plane_pointclouds_to_meshes(combined_pcds, filtered_meshes,voxel_size=VOXEL_SIZE,scanner_center= translate_matrix[:3, 3])
scene = drm.visualise_open3d(projected_meshes, random_color=True)
scene.show()

### UV Generation

To properly inpaint the generated planes, we need uv's per plane. It is important that the projected plane is oriented correctly so we aim the normal towards the sensor origin. each plane is mapped, keeping the world up direction the same as the uv up. and the front facing direction as the normal direction. The plane is mapped to the full 0-1 range of the uv map

In [ ]:
uv_meshes = drm.generate.assign_plane_uvs(projected_meshes, pcd.get_axis_aligned_bounding_box().get_center())
for mesh in uv_meshes:
    mesh.compute_triangle_normals()
textured_meshes = drm.generate.apply_texture_to_planes(uv_meshes, r"/home/jvermandere/projects/DRM/_input/UV_Grid_Sm.jpg")
scene = drm.visualise_open3d(textured_meshes)
scene.lights.append(
    trimesh.scene.lighting.PointLight(
        color=[255, 255, 255, 255],
        intensity=100.0,
        radius=0.0,
    )
)
scene.show()

### Pano Projection

To get the best results, if it is available, we use the pano image and project it on each plane of the scene. but of course, there are still a bunch of objects visible in the pano so we need to paint them out. to get the best results, we will paint them out in the projected image per plane. Therefore we need a 

In [ ]:
from PIL import Image
Image.open(scene_dir / PANO_FILE)

In [ ]:
pano_image = np.array(Image.open(scene_dir / PANO_FILE).convert("RGB"))
trans_matrix = drm.read_transform_matrix(scene_dir / "main.txt", apply_unity_conversion=True)
annotated = drm.generate.draw_plane_corners_on_pano(uv_meshes[2], pano_image, trans_matrix)
Image.fromarray(annotated)

In [ ]:
pano_image = np.array(Image.open(scene_dir / PANO_FILE).convert("RGB"))
trans_matrix = drm.read_transform_matrix(scene_dir / "main.txt", apply_unity_conversion=True)

textured   = drm.generate.project_pano_onto_planes(uv_meshes, pano_image, trans_matrix)
scene      = drm.visualise_open3d(textured)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

In [ ]:
pano_image = np.array(Image.open(scene_dir / EMPTY_PANO_FILE).convert("RGB"))
trans_matrix = drm.read_transform_matrix(scene_dir / "main.txt", apply_unity_conversion=True)

empty_textured   = drm.generate.project_pano_onto_planes(uv_meshes, pano_image, trans_matrix)
scene      = drm.visualise_open3d(empty_textured)
scene.add_geometry(trimesh.creation.axis(0.1))
scene.show()

### Occlusion mask generation
We generate an occlusion mask using the occupied and occlused voxel grid. From the pano image, we check what parts of the image do not lie on the created planes's depth. Those pixels are masked to be painted in later. This can be sampled directly from the occlusion grid if the resolution is fine enough, or from a (reconstructed from the source pcd) depth pano and comparing it to the empty depth image generated from the newly generated planes.

In [ ]:


from PIL import Image
EMPTY_PANO_FILE = "pano_empty.png"
pano_image       = np.array(Image.open(scene_dir / PANO_FILE).convert("RGB"))
empty_pano_image = np.array(Image.open(scene_dir / EMPTY_PANO_FILE).convert("RGB"))

masked = drm.generate.generate_occlusion_mask_image(pano_image, empty_pano_image, threshold=3, blur_radius=5.0)
maskedPanoImage = Image.fromarray(masked)   # transparent where objects occlude the empty scene
maskedPanoImage

In [ ]:
masked_plane_textures = drm.generate.sample_pano_textures(uv_meshes, maskedPanoImage, trans_matrix)
Image.fromarray(masked_plane_textures[1])

### Inpainting

The missing regions are inpainted on each plane seperatly. using simpleLaMa where the masks are dilated by 10px to ensure no parts of the clutter or moveable furniture remains on the image.

In [ ]:
inpainted_textures = drm.generate.inpaint_plane_textures(masked_plane_textures, dilation_px=10)                  # RGB


In [ ]:
Image.fromarray(inpainted_textures[1])

### Apply textures to mesh
As the final step, the textures are applied to the mesh.

In [ ]:

# Apply to meshes
textured_meshes = drm.generate.apply_texture_to_planes(uv_meshes, inpainted_textures)
scene = drm.visualise_open3d(textured_meshes)
scene.add_geometry(trimesh.creation.axis(0.1))

scene.export("scene.glb")  # open in Blender, three.js, or any glTF viewer

## Dataset Processing

In [ ]:
import sys
sys.path.insert(0, '../')
import drm
import drm.pipeline

def reconstruct_dataset(dataset_root: str, skip_existing: bool = True) -> None:
    """
    Run reconstruct_scene on every scan folder in the dataset.

    Parameters
    ----------
    dataset_root : Path to the root folder containing scan subfolders.
    skip_existing : Whether to skip folders that already have reconstructed environments.
    """
    from pathlib import Path
    from simple_lama_inpainting import SimpleLama

    dataset_root = Path(dataset_root)
    scan_folders = sorted([p for p in dataset_root.iterdir() if p.is_dir()])

    print(f"Found {len(scan_folders)} scan folders.")

    # Load once, reuse across all scenes
    print("Loading SimpleLama model...")
    simple_lama = SimpleLama()

    for i, folder in enumerate(scan_folders):
        print(f"\n[{i+1}/{len(scan_folders)}] {folder.name}")

        # Skip if already processed
        if skip_existing and (folder / "reconstructed_environment.glb").exists():
            print("  Already processed, skipping.")
            continue

        # Skip folders missing required files
        required = ["main.txt", "pano.png", "pano_empty.png", "main_bb.json"]
        missing  = [f for f in required if not (folder / f).exists()]
        if missing:
            print(f"  Missing files: {missing}, skipping.")
            continue

        try:
            drm.pipeline.reconstruct_scene(folder, simple_lama=simple_lama)
        except Exception as e:
            print(f"  ERROR processing {folder.name}: {e}")
            import traceback
            traceback.print_exc()
            continue

    print("\nDone.")


reconstruct_dataset("/home/jvermandere/datasets/V-Scan/data", skip_existing=True)

## V-Scan Evaluation

In [ ]:
def evaluate_geometry(
    reconstructed_meshes: list[o3d.geometry.TriangleMesh],
    ground_truth_pcd:     o3d.geometry.PointCloud,
) -> dict:
    """
    Evaluate geometric accuracy of reconstructed plane meshes against
    a ground truth point cloud.

    Metrics
    -------
    - Chamfer Distance      : mean bidirectional point-to-surface distance
    - Hausdorff Distance    : worst-case deviation
    - F-Score @ thresholds  : % of points within distance threshold (precision/recall)
    """
    from scipy.spatial import cKDTree

    gt_points  = np.asarray(ground_truth_pcd.points)
    gt_tree    = cKDTree(gt_points)

    # Sample points from reconstructed meshes
    recon_pcd  = o3d.geometry.PointCloud()
    for mesh in reconstructed_meshes:
        if len(mesh.triangles) == 0:
            continue
        sampled = mesh.sample_points_uniformly(
            number_of_points=max(1000, int(mesh.get_surface_area() / 0.001))
        )
        recon_pcd += sampled

    recon_points = np.asarray(recon_pcd.points)
    recon_tree   = cKDTree(recon_points)

    # Recon → GT distances
    d_recon_to_gt, _ = gt_tree.query(recon_points,   k=1)
    # GT → Recon distances
    d_gt_to_recon, _ = recon_tree.query(gt_points,   k=1)

    chamfer    = (d_recon_to_gt.mean() + d_gt_to_recon.mean()) / 2
    hausdorff  = max(d_recon_to_gt.max(), d_gt_to_recon.max())

    # F-Score at multiple thresholds
    f_scores = {}
    for threshold in [0.01, 0.02, 0.05, 0.10]:
        precision = (d_recon_to_gt < threshold).mean()
        recall    = (d_gt_to_recon < threshold).mean()
        denom     = precision + recall
        f         = 2 * precision * recall / denom if denom > 0 else 0.0
        f_scores[f"f_score@{int(threshold*100)}cm"] = float(f)
        f_scores[f"precision@{int(threshold*100)}cm"] = float(precision)
        f_scores[f"recall@{int(threshold*100)}cm"]    = float(recall)

    return {
        "chamfer_distance_m":  float(chamfer),
        "hausdorff_distance_m": float(hausdorff),
        **f_scores,
        "n_recon_points": len(recon_points),
        "n_gt_points":    len(gt_points),
    }

def evaluate_texture(
    uv_meshes:          list[o3d.geometry.TriangleMesh],
    inpainted_textures: list[np.ndarray],
    empty_pano_image:   np.ndarray,
    trans_matrix:       np.ndarray,
    loss_fn=None,
    device:             str = "cpu",
) -> dict:
    import torch
    from skimage.metrics import peak_signal_noise_ratio, structural_similarity
    from PIL import Image

    if loss_fn is None:
        import lpips
        loss_fn = lpips.LPIPS(net="alex").eval().to(device)

    gt_textures = drm.generate.sample_pano_textures(
        uv_meshes, empty_pano_image, trans_matrix
    )

    per_plane, psnr_all, ssim_all, lpips_all = [], [], [], []

    for i, (pred, gt) in enumerate(zip(inpainted_textures, gt_textures)):
        if pred.shape != gt.shape:
            gt = np.array(Image.fromarray(gt).resize(
                (pred.shape[1], pred.shape[0]), Image.BILINEAR
            ))

        pred_f = pred.astype(np.float32) / 255.0
        gt_f   = gt.astype(np.float32)   / 255.0

        psnr = peak_signal_noise_ratio(gt_f, pred_f, data_range=1.0)
        ssim = structural_similarity(gt_f, pred_f, channel_axis=2, data_range=1.0)

        def to_lpips_tensor(img_f):
            t = torch.from_numpy(img_f).permute(2, 0, 1).unsqueeze(0).float()
            return (t * 2.0 - 1.0).to(device)

        with torch.no_grad():
            lp = float(loss_fn(to_lpips_tensor(pred_f), to_lpips_tensor(gt_f)).item())

        psnr_all.append(psnr);  ssim_all.append(ssim);  lpips_all.append(lp)
        per_plane.append({"plane": i, "psnr": psnr, "ssim": ssim, "lpips": lp})
        print(f"  Plane {i}: PSNR={psnr:.2f}dB  SSIM={ssim:.4f}  LPIPS={lp:.4f}")

    return {
        "mean_psnr":  float(np.mean(psnr_all)),
        "mean_ssim":  float(np.mean(ssim_all)),
        "mean_lpips": float(np.mean(lpips_all)),
        "per_plane":  per_plane,
    }

def evaluate_dataset(
    dataset_root: str,
    output_csv:   str = "evaluation_results.csv",
) -> None:
    import csv
    import torch
    import lpips
    from pathlib import Path
    from PIL import Image

    dataset_root = Path(dataset_root)
    output_csv   = Path(output_csv)

    # --- load already completed scenes from existing CSV ---
    completed = set()
    existing_rows = []
    if output_csv.exists():
        with open(output_csv, "r", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                existing_rows.append(row)
                completed.add(row["scene"])
        print(f"Found {len(completed)} already evaluated scenes, skipping.")

    # --- load LPIPS once ---
    print("Loading LPIPS model...")
    loss_fn = lpips.LPIPS(net="alex").eval()
    device  = "cuda" if torch.cuda.is_available() else "cpu"
    loss_fn = loss_fn.to(device)

    scan_folders = sorted([p for p in dataset_root.iterdir() if p.is_dir()])
    fieldnames   = None   # will be set on first write

    for i, folder in enumerate(scan_folders):
        glb_path = folder / "reconstructed_environment.glb"
        if not glb_path.exists():
            continue

        if folder.name in completed:
            print(f"[{i+1}] Skipping (already done): {folder.name}")
            continue

        print(f"\n[{i+1}] Evaluating: {folder.name}")

        try:
            # --- ground truth ---
            gt_pcd, _    = drm.txt_pcd_to_open3d(folder / "main_empty.txt")
            empty_pano   = np.array(Image.open(folder / "pano_empty.png").convert("RGB"))
            trans_matrix = drm.read_transform_matrix(
                folder / "main.txt", apply_unity_conversion=True
            )

            # --- load GLB ---
            scene              = trimesh.load(str(glb_path), force="scene")
            uv_meshes_o3d      = []
            inpainted_textures = []
            all_meshes         = []

            for g in scene.geometry.values():
                if not isinstance(g, trimesh.Trimesh) or len(g.faces) == 0:
                    continue

                m = o3d.geometry.TriangleMesh()
                m.vertices  = o3d.utility.Vector3dVector(np.array(g.vertices,  dtype=np.float64))
                m.triangles = o3d.utility.Vector3iVector(np.array(g.faces,     dtype=np.int32))
                all_meshes.append(m)

                uv, tex_img = None, None
                try:
                    if hasattr(g.visual, "uv") and g.visual.uv is not None:
                        uv           = np.array(g.visual.uv, dtype=np.float64)
                        triangle_uvs = uv[np.array(g.faces).flatten()]
                        m.triangle_uvs = o3d.utility.Vector2dVector(triangle_uvs)
                except Exception as e:
                    print(f"  UV extraction failed: {e}")

                try:
                    if hasattr(g.visual, "material"):
                        mat = g.visual.material
                        img = getattr(mat, "image", None) or getattr(mat, "baseColorTexture", None)
                        if img is not None:
                            tex_img = np.ascontiguousarray(np.array(img.convert("RGB")))
                except Exception as e:
                    print(f"  Texture extraction failed: {e}")

                if uv is not None and tex_img is not None:
                    m.textures = [o3d.geometry.Image(tex_img)]
                    m.triangle_material_ids = o3d.utility.IntVector(
                        np.zeros(len(g.faces), dtype=np.int32)
                    )
                    uv_meshes_o3d.append(m)
                    inpainted_textures.append(tex_img)

            print(f"  Loaded {len(uv_meshes_o3d)} textured meshes from GLB")

            # --- evaluate ---
            geo_metrics = evaluate_geometry(all_meshes, gt_pcd)

            if uv_meshes_o3d and inpainted_textures:
                tex_metrics = evaluate_texture(
                    uv_meshes_o3d, inpainted_textures,
                    empty_pano, trans_matrix,
                    loss_fn=loss_fn, device=device,
                )
            else:
                print("  WARNING: no textured meshes found, skipping texture metrics")
                tex_metrics = {
                    "mean_psnr":  float("nan"),
                    "mean_ssim":  float("nan"),
                    "mean_lpips": float("nan"),
                }

            row = {
                "scene":      folder.name,
                **geo_metrics,
                "mean_psnr":  tex_metrics["mean_psnr"],
                "mean_ssim":  tex_metrics["mean_ssim"],
                "mean_lpips": tex_metrics["mean_lpips"],
            }

            # --- append to CSV immediately ---
            if fieldnames is None:
                fieldnames = list(row.keys())

            write_header = not output_csv.exists() or output_csv.stat().st_size == 0
            with open(output_csv, "a", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if write_header:
                    writer.writeheader()
                    # re-write any rows loaded from a previous incomplete run
                    for r in existing_rows:
                        writer.writerow(r)
                writer.writerow(row)

            completed.add(folder.name)
            print(f"  Chamfer: {geo_metrics['chamfer_distance_m']*100:.2f}cm  "
                  f"PSNR: {tex_metrics['mean_psnr']:.1f}dB  "
                  f"SSIM: {tex_metrics['mean_ssim']:.3f}  "
                  f"LPIPS: {tex_metrics['mean_lpips']:.3f}")

        except Exception as e:
            print(f"  ERROR: {e}")
            import traceback; traceback.print_exc()

    print(f"\nDone. Results saved to {output_csv}")

evaluate_dataset("/home/jvermandere/datasets/V-Scan/data", output_csv="evaluation_results.csv")

## Matterport Experiments

In [ ]:
import numpy as np
import open3d as o3d
import trimesh
import os
import sys
sys.path.insert(0, '../')
import drm
import drm.detect
import drm.align
import drm.generate
import drm.pipeline
from PIL import Image
import torch
import numpy as np
from pathlib import Path

ply_path = Path(r"/home/jvermandere/datasets/MatterPort3D/selection/1pXnuDYAj8r__0f42320ef887416fb8ed9b91b6641cfc/0f42320ef887416fb8ed9b91b6641cfc.ply")
pcd = o3d.io.read_point_cloud(str(ply_path))

drm.visualise_open3d([pcd]).show()    

# Old Scene Completion

## Load pointcloud

In [ ]:
pcdPath = "/home/jvermandere/projects/DRM/_input/virtualDataset/VirtualScanner-1773153754053/results/segmented_points/isolated_points.txt"
pointsColors = np.loadtxt(pcdPath, dtype=np.float32).reshape(-1, 6)
cloud = trimesh.points.PointCloud(pointsColors[:, :3], colors=pointsColors[:, 3:6]/255)
scene = trimesh.Scene(cloud)
scene.show()

## Ransac itterative plane detection

In [ ]:
min_points = 100  # minimum points to consider a plane
remaining_points = cloud.vertices.copy()
remaining_colors = cloud.colors.copy() if cloud.colors is not None else None

planes = []  # list of trimesh point clouds for each plane

while len(remaining_points) >= min_points:
    plane_model, inliers = drm.detect.ransac_plane_trimesh(remaining_points,
                                        num_iterations=1000,
                                        distance_threshold=0.01)

    if len(inliers) < min_points:
        print("No more large planes detected.")
        break

    # Extract plane points and colors
    plane_pts = remaining_points[inliers]
    plane_colors = remaining_colors[inliers] if remaining_colors is not None else None
    plane_pc = trimesh.points.PointCloud(vertices=plane_pts, colors=plane_colors)
    planes.append(plane_pc)

    # Remove plane points from remaining points
    mask = np.ones(len(remaining_points), dtype=bool)
    mask[inliers] = False
    remaining_points = remaining_points[mask]
    if remaining_colors is not None:
        remaining_colors = remaining_colors[mask]

# Remaining points as a separate point cloud
if len(remaining_points) > 0:
    remaining_pc = trimesh.points.PointCloud(vertices=remaining_points,
                                             colors=remaining_colors)
else:
    remaining_pc = None

print(f"Extracted {len(planes)} planes.")

In [ ]:
scene = drm.visualize_pointclouds_random_colors(planes)
scene.show()

## Inpainting all the planes

In [ ]:
from simple_lama_inpainting import SimpleLama
simple_lama = SimpleLama()
# Paint in all the planes one by one
filled_planes = []
for i in range(len(planes)):
    #fill in the planes
    filled_plane = drm.fill_plane_holes(planes[i], target_density=0.01)
    # get the 2D images of the partial plane and to-be-filled-in area of the new plane
    img, mask, filled_coords = drm.project_planes_with_infill_mask(planes[i], filled_plane, resolution=512, point_radius=1)
    # paint in the image
    image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB").resize((512, 512))
    mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L").resize((512, 512))
    # Run inpainting
    inpaintedImage = simple_lama(image_pil,mask_pil)
    #
    texture = np.asarray(inpaintedImage)
    colors = texture[filled_coords[:,1], filled_coords[:,0]]
    filled_plane.colors = np.hstack([colors,np.full((len(colors),1),255)])
    filled_planes.append(filled_plane)


In [ ]:
scene = trimesh.Scene(filled_planes)
scene.show()

### Save the completed pointcloud

In [ ]:
all_vertices = []
all_colors = []

for pc in filled_planes[:4]:
    all_vertices.append(pc.vertices)

    if pc.colors is not None:
        all_colors.append(pc.colors)
    else:
        # default white
        all_colors.append(np.full((len(pc.vertices), 4), 255, dtype=np.uint8))

vertices = np.vstack(all_vertices)
colors = np.vstack(all_colors)

merged_pc = trimesh.points.PointCloud(vertices=vertices, colors=colors)

merged_pc.export(Path(pcdPath).as_posix()[:-4] + "_reconstructed.ply")

## Inpainting one plane

In [ ]:
filled_planes = []
for plane in planes:
    filled_plane = drm.fill_plane_holes(plane, target_density=0.01)
    filled_planes.append(filled_plane)

scene = trimesh.Scene(filled_planes[:4])
scene.show()

In [ ]:
import matplotlib.pyplot as plt

# filled_plane = fill_plane_holes_with_colors(plane)
img, mask, filled_coords = drm.project_planes_with_infill_mask(planes[3], filled_planes[3], resolution=512, point_radius=1)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(img)
plt.title("Original Plane Colors")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Filled Areas / Holes")
plt.axis("off")

plt.show()

### StableDiffusion Inpainting

In [ ]:
from PIL import Image
import torch
from diffusers import StableDiffusionInpaintPipeline
import numpy as np
# Load inpainting model
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting"
).to("cuda")



In [ ]:

# Convert numpy arrays
image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB")
mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")

# Resize both to same dimensions (512x512 is safe for SD)
image_pil = image_pil.resize((512, 512))
mask_pil = mask_pil.resize((512, 512))



#prompt = "paint in the masked area to fill in the missing regions looking at the rest of the colors in the image to match the texture, ignore eveything that is black"
prompt = "paint"

# Run inpainting
result = pipe(
    prompt=prompt,
    image=image_pil,
    mask_image=mask_pil
).images[0]


In [ ]:
result

### LaMa

In [ ]:
from simple_lama_inpainting import SimpleLama
from PIL import Image

simple_lama = SimpleLama()

# Convert numpy arrays
image_pil = Image.fromarray(img.astype(np.uint8)).convert("RGB")
mask_pil = Image.fromarray((mask * 255).astype(np.uint8)).convert("L")
image_pil = image_pil.resize((512, 512))
mask_pil = mask_pil.resize((512, 512))

result = simple_lama(image_pil, mask_pil)
result

### Reprojection

In [ ]:
def apply_texture_to_plane(filled_pc, inpainted_img, filled_coords):

    img = np.asarray(inpainted_img)

    new_colors = np.zeros((len(filled_coords), 4), dtype=np.uint8)

    for i, (x,y) in enumerate(filled_coords):

        color = img[y, x]

        if len(color) == 3:
            new_colors[i] = np.append(color, 255)
        else:
            new_colors[i] = color

    filled_pc.colors = new_colors

    return filled_pc

In [ ]:
filled_pc = apply_texture_to_plane(
    filled_planes[3],
    result,
    filled_coords
)

In [ ]:
scene = trimesh.Scene(filled_pc)
scene.show()

In [ ]:
from simple_lama_inpainting import SimpleLama
from PIL import Image

simple_lama = SimpleLama()

image_pil = image_pil.resize((512, 512))
mask_pil = mask_pil.resize((512, 512))

result = simple_lama(image_pil, mask_pil)
result